# 06 · Spatial epidemiology: time, place, denominator


**Mission (40 minutes):** turn aggregate first-episode reports into an epidemic curve, a choropleth, and an uncertainty-aware comparison. The illustrative case definition is one newly reported first episode of the exercise syndrome per resident during the 14-day window. It is not a diagnostic definition.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

## 1. Plot by onset date, and mark incomplete days
These are aggregated reported cases by onset date. No individual line list or patient coordinates are used. The right edge has deliberately incomplete reporting; a drop there cannot be read as control of an outbreak.

In [ ]:
rows = load_cases()
dates = sorted({r['onset_date'] for r in rows})
daily = np.array([sum(r['cases'] for r in rows if r['onset_date']==d) for d in dates])
fig, ax = plt.subplots(figsize=(11,4), constrained_layout=True)
ax.bar(np.arange(14), daily, width=1, color=['#20a89d']*11+['#e3b258']*3, edgecolor='white')
ax.axvspan(10.5,13.5,color='#e3b258',alpha=.14,label='Incomplete reporting')
ax.set_xticks(range(0,14,2), [d[5:] for d in dates[::2]])
ax.set(xlabel='Onset date (2026)',ylabel='Reported first episodes',title='Synthetic epidemic curve · fixed 15 Sep snapshot')
ax.legend(); plt.show()
assert daily.sum() == sum(s['cases'] for s in stats)

## 2. Rates and intervals
For this closed fictional population, reported cumulative incidence per 10,000 is `cases / population × 10,000`. A Wilson binomial interval illustrates sampling uncertainty under an independent first-episode assumption. It **does not account for reporting delay, under-ascertainment, migration, spatial dependence, or a wrong denominator**.

In [ ]:
k = np.array([s['cases'] for s in stats]); n = np.array([s['population'] for s in stats])
r = rate_per_10000(k,n); low, high = wilson_interval(k,n)
order = np.argsort(r)
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
p = plot_sectors(axes[0],r,'Synthetic reported cumulative incidence')
fig.colorbar(p,ax=axes[0],shrink=.7,label='Reported first episodes / 10,000 / 14 days')
axes[1].errorbar(r[order],np.arange(12),xerr=[r[order]-low[order]*10000,high[order]*10000-r[order]],fmt='o',color='#158b85')
axes[1].set_yticks(range(12),[stats[i]['id'] for i in order])
axes[1].set(xlabel='Reported cases / 10,000 (95% Wilson interval)',ylabel='Training sector',title='Intervals omit systematic errors')
plt.show()
print('Missing population yields:', float(rate_per_10000(10,0)), '(not zero risk)')

## 3. Sensitivity to ascertainment
Hypothetical reporting probabilities are scenario assumptions, not estimates. Dividing by them is an arithmetic sensitivity exercise, not a validated nowcast.

In [ ]:
for reporting in [.4,.7,1.0]:
    adjusted = k.sum()/reporting
    print(f'If reporting probability = {reporting:.0%}: implied episodes ≈ {adjusted:,.0f}')

## Lab challenge and checks
1. Double S07's population without changing its reports. Its rate should halve.
2. Compare count and rate rankings. Which would you show for staffing workload versus population comparison?
3. Explain why a cluster around the invented river is not proof of waterborne transmission.

**Check:** an onset curve and a report-date curve answer different questions. Population denominators and case definitions must match in space, time, and eligibility. An ecological association cannot identify an individual's exposure or establish a cause.

**Sources:** [CDC: Describing Epidemiologic Data](https://www.cdc.gov/field-epi-manual/php/chapters/describing-epi-data.html), [CDC: Analyzing and Interpreting Data](https://www.cdc.gov/field-epi-manual/php/chapters/analyze-interpret-data.html). Synthetic data generation: `scripts/generate_data.py`; no real surveillance data are redistributed.